# Nettoyage du domaine écologie

Le domaine écologie contient six fichiers sur des sites naturels suivis pendant plusieurs années : des grottes, des forêts et des vergers de durians. Avant de chercher quoi que ce soit dans ces données, on veut être sûrs qu'elles sont propres : pas de lignes en double, des nombres lisibles, des dates dans un seul format et des catégories écrites de la même façon.

On regarde d'abord ce que contient chaque fichier, puis on traite les problèmes un par un. Pour chaque problème, on montre ce qu'on a trouvé avant de le corriger. À la fin, on enregistre les fichiers propres dans `data_clean/ecology/`.

Ce notebook remplace l'ancien `04_ecology_nettoyage.ipynb`, qui appliquait une boucle générique à tous les fichiers sans regarder leur contenu.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

SOURCE = Path("data/ecology")
SORTIE = Path("data_clean/ecology")

# On lit tout comme du texte : si une colonne contient « 9 count » au milieu de nombres,
# pandas la lirait sinon comme du texte sans prévenir, ou transformerait certains codes.
# En partant du texte, on garde la main sur chaque conversion.
noms = ["monitoring_sites", "bat_surveys", "environmental_monthly",
        "monitoring_device_operations", "orchard_surveys", "environmental_field_reports"]
tables = {nom: pd.read_csv(SOURCE / f"{nom}.csv", dtype=str) for nom in noms}

for nom, df in tables.items():
    print(f"{nom:30s} {df.shape[0]:5d} lignes  {df.shape[1]:2d} colonnes")

monitoring_sites                  64 lignes  14 colonnes
bat_surveys                     4994 lignes  14 colonnes
environmental_monthly           4994 lignes  11 colonnes
monitoring_device_operations    4994 lignes  11 colonnes
orchard_surveys                 1250 lignes  11 colonnes
environmental_field_reports       30 lignes  12 colonnes


## 1. Que contient chaque fichier ?

- `monitoring_sites` : la liste des 64 sites suivis, avec leur région, leur type d'habitat et, pour les grottes, leur catégorie d'accès.
- `bat_surveys` : les relevés de chauves-souris, à peu près un par site et par mois. On y trouve le nombre de chauves-souris observées, les détections acoustiques, une estimation de la colonie et la qualité du relevé.
- `environmental_monthly` : la météo et l'état du milieu chaque mois et sur chaque site (température, pluie, végétation, perturbation, pesticides).
- `monitoring_device_operations` : le fonctionnement des appareils d'écoute chaque mois (temps de marche, qualité du son, pannes).
- `orchard_surveys` : les relevés des vergers de durians (fleurs, fruits, récolte).
- `environmental_field_reports` : 30 rapports écrits par des équipes de terrain.

On regarde quelques lignes du fichier principal, celui des chauves-souris.

In [2]:
tables["bat_surveys"].head()

,survey_id,site_code,survey_start,survey_end,survey_method,observation_hours,observed_bat_count,acoustic_detection_count,nightly_visit_count,colony_estimate,comparison_species_count,observer_team,survey_quality,weather_interference
0,BAT-0000001,KNT-KES,2019-01-05T18:00:00,2019-01-12T00:00:00,fixed_acoustic_station,28.6,7,13,7,3.8,8,TEAM-11-2,high,32.0
1,BAT-0000002,KNT-KES,2019-02-05T18:00:00,2019-02-12T00:00:00,acoustic_and_visual,31.7,8,12,9,7.3,18,TEAM-11-2,acceptable,18.5
2,BAT-0000003,KNT-KES,2019-03-05T18:00:00,2019-03-12T00:00:00,fixed_acoustic_station,32.4,22,28,14,19.9,8,TEAM-11-2,high,26.2
3,BAT-0000004,KNT-KES,2019-04-05T18:00:00,2019-04-12T00:00:00,acoustic_and_visual,35.7,26,36,19,32.2,14,TEAM-11-2,high,35.0
4,BAT-0000005,KNT-KES,2019-05-05T18:00:00,2019-05-12T00:00:00,acoustic_transect,32.0,23,29,22,20.9,16,TEAM-11-2,acceptable,38.7


## 2. Les lignes en double

Une ligne en double compterait deux fois le même relevé. On compte, pour chaque fichier, les lignes entièrement identiques et les identifiants qui reviennent plusieurs fois.

In [3]:
cles = {"monitoring_sites": ["site_code"],
        "bat_surveys": ["survey_id"],
        "environmental_monthly": ["site_code", "reporting_month"],
        "monitoring_device_operations": ["device_operation_id"],
        "orchard_surveys": ["orchard_survey_id"],
        "environmental_field_reports": ["document_id"]}

pd.DataFrame({nom: {"lignes identiques": df.duplicated().sum(),
                    "identifiants répétés": df.duplicated(cles[nom]).sum()}
              for nom, df in tables.items()}).T

,lignes identiques,identifiants répétés
monitoring_sites,0,0
bat_surveys,2,2
environmental_monthly,2,2
monitoring_device_operations,2,2
orchard_surveys,2,2
environmental_field_reports,0,0


Quatre fichiers ont exactement 2 lignes en double, et à chaque fois les identifiants répétés sont ceux de ces lignes. Ce sont donc de simples copies : on les supprime. Pour `environmental_monthly`, qui n'a pas d'identifiant, on a utilisé le couple site et mois.

In [4]:
for nom in tables:
    tables[nom] = tables[nom].drop_duplicates().reset_index(drop=True)
    assert not tables[nom].duplicated(cles[nom]).any()

## 3. Les nombres mal écrits

Dans plusieurs colonnes censées contenir des nombres, certaines valeurs ne sont pas lisibles comme des nombres. On les affiche pour voir de quoi il s'agit.

In [5]:
def valeurs_non_numeriques(df):
    resultat = {}
    for c in df.columns:
        v = df[c].dropna()
        lisibles = pd.to_numeric(v, errors="coerce").notna()
        if 0.5 < lisibles.mean() < 1:          # surtout des nombres, mais pas que
            resultat[c] = v[~lisibles].unique()[:4].tolist()
    return resultat

for nom, df in tables.items():
    for colonne, exemples in valeurs_non_numeriques(df).items():
        print(f"{nom:30s} {colonne:26s} {exemples}")

bat_surveys                    observation_hours          ['24,3', '37,2', '30,6', '35,0']
bat_surveys                    observed_bat_count         ['9 count', '11 count', '16 count', '18 count']
bat_surveys                    acoustic_detection_count   ['7 count', '52 count', '47 count', '13 count']
bat_surveys                    nightly_visit_count        ['7 count', '1 count', '5 count', '14 count']
bat_surveys                    weather_interference       ['15,0', '42,4', '7,7', '34,1']
environmental_monthly          mean_temperature_c         ['23,81', '26,34', '29,35', '26,63']
environmental_monthly          rainfall_mm                ['102.7 mm', '75,5', '251.3 mm', '67.9 mm']
environmental_monthly          mean_humidity_pct          ['74,8', '64,7', '73,5', '83,2']


monitoring_device_operations   uptime_pct                 ['99,3', '91,3', '98,1', '90,9']
orchard_surveys                orchard_area_ha            ['59.7 ha', '54.7 ha', '6.0 ha', '31.3 ha']
orchard_surveys                durian_flower_density      ['48,4', '49,7', '49,5', '71,2']
orchard_surveys                durian_yield_kg            ['63290.0 kg', '69487.6 kg', '116624.0 kg', '7641.7 kg']
orchard_surveys                fertilizer_use_index       ['50,1', '46,8', '44,4', '29,8']


On voit deux sortes d'erreurs :
- une virgule à la place du point (« 24,3 ») ;
- une unité collée au nombre (« 9 count », « 102.7 mm », « 59.7 ha », « 63290.0 kg »).

Dans les deux cas, le nombre est bon, c'est juste son écriture qui change. On remplace la virgule par un point et on garde le nombre qui se trouve au début. On n'a pas besoin de garder l'unité, puisqu'elle est déjà donnée par le nom de la colonne (`rainfall_mm`, `durian_yield_kg`…).

In [6]:
def en_nombre(serie):
    texte = serie.astype("string").str.strip().str.replace(",", ".", regex=False)
    nombre = texte.str.extract(r"^(-?\d+(?:\.\d+)?)")[0]
    return pd.to_numeric(nombre, errors="coerce")

colonnes_numeriques = {
    "monitoring_sites": ["latitude", "longitude", "elevation_m"],
    "bat_surveys": ["observation_hours", "observed_bat_count", "acoustic_detection_count",
                    "nightly_visit_count", "colony_estimate", "comparison_species_count",
                    "weather_interference"],
    "environmental_monthly": ["mean_temperature_c", "rainfall_mm", "mean_humidity_pct",
                              "vegetation_cover_pct", "habitat_disturbance_index",
                              "pesticide_index", "data_completeness_pct"],
    "monitoring_device_operations": ["uptime_pct", "acoustic_quality"],
    "orchard_surveys": ["orchard_area_ha", "durian_flower_density", "fruit_set_rate_pct",
                        "durian_yield_kg", "other_pollinator_activity",
                        "fertilizer_use_index", "irrigation_level"],
}

for nom, colonnes in colonnes_numeriques.items():
    for c in colonnes:
        avant = tables[nom][c].notna().sum()
        tables[nom][c] = en_nombre(tables[nom][c])
        perdues = avant - tables[nom][c].notna().sum()
        if perdues:
            print("valeurs perdues :", nom, c, perdues)
print("Conversion terminée")

Conversion terminée


Aucune valeur n'a été perdue pendant la conversion : chaque texte contenait bien un nombre.

### Des valeurs dans la bonne échelle ?

Un nombre lisible n'est pas forcément juste. On regarde le minimum et le maximum de chaque colonne pour repérer des valeurs impossibles, par exemple un pourcentage au-dessus de 100 ou un comptage négatif.

In [7]:
for nom, colonnes in colonnes_numeriques.items():
    print(nom)
    print(tables[nom][colonnes].describe().loc[["min", "50%", "max"]].round(2).to_string(), "\n")

monitoring_sites
     latitude  longitude  elevation_m
min      1.77     101.33          5.0
50%      4.74     104.46        142.4
max      7.76     112.69        418.8 

bat_surveys
     observation_hours  observed_bat_count  acoustic_detection_count  nightly_visit_count  colony_estimate  comparison_species_count  weather_interference
min                5.0                 0.0                       0.0                  0.0              0.0                       1.0                   0.0
50%               30.6                13.0                      18.0                  9.0             14.2                      11.0                  36.7
max               46.9                73.0                     107.0                 54.0             99.6                      26.0                 100.0 

environmental_monthly


     mean_temperature_c  rainfall_mm  mean_humidity_pct  vegetation_cover_pct  habitat_disturbance_index  pesticide_index  data_completeness_pct
min                21.3          5.0               56.7                  15.0                        0.0              0.0                   85.5
50%               26.35       156.25               75.9                  62.3                       29.4             25.2                  96.15
max                32.0        384.3               96.4                  98.0                       86.8             77.3                  100.0 

monitoring_device_operations
     uptime_pct  acoustic_quality
min        23.1              20.0
50%        93.8              79.3
max       100.0             100.0 

orchard_surveys
     orchard_area_ha  durian_flower_density  fruit_set_rate_pct  durian_yield_kg  other_pollinator_activity  fertilizer_use_index  irrigation_level
min              6.0                   17.0                0.47           3971.3       

Presque tout est plausible : les pourcentages restent entre 0 et 100, et les comptages ne sont jamais négatifs.

Une colonne attire l'attention : `fruit_set_rate_pct`, le pourcentage de fleurs qui deviennent des fruits. Sa médiane est d'environ 53 %, mais son minimum vaut 0,47. On regarde les petites valeurs.

In [8]:
nouaison = tables["orchard_surveys"]["fruit_set_rate_pct"]
print("Valeurs inférieures à 1 :", nouaison[nouaison < 1].tolist())
print("Plus petite valeur au-dessus de 1 :", nouaison[nouaison >= 1].min())

Valeurs inférieures à 1 : [0.568, 0.543, 0.549, 0.561, 0.595, 0.467, 0.581, 0.555]
Plus petite valeur au-dessus de 1 : 27.3


Huit valeurs sont comprises entre 0,47 et 0,60, puis il n'y a plus rien jusqu'à 27. Un taux de 0,5 % serait une récolte presque nulle, alors que les autres colonnes de ces mêmes lignes (fleurs, récolte en kg) sont normales. Ces huit valeurs ont donc été écrites comme des fractions : 0,568 veut dire 56,8 %. On les multiplie par 100.

In [9]:
petites = tables["orchard_surveys"]["fruit_set_rate_pct"] < 1
tables["orchard_surveys"].loc[petites, "fruit_set_rate_pct"] *= 100
tables["orchard_surveys"]["fruit_set_rate_pct"].describe().round(1)

count    1248.0
mean       53.6
std         9.3
min        27.3
25%        47.0
50%        53.0
75%        59.4
max        87.7
Name: fruit_set_rate_pct, dtype: Float64

## 4. Les dates écrites de plusieurs façons

La plupart des dates suivent le format `2019-01-12T00:00:00`. On cherche celles qui sont écrites autrement.

In [10]:
colonnes_dates = {
    "monitoring_sites": ["programme_start_date", "programme_end_date"],
    "bat_surveys": ["survey_start", "survey_end"],
    "environmental_monthly": ["reporting_month"],
    "monitoring_device_operations": ["operation_start", "operation_end"],
    "orchard_surveys": ["survey_period"],
    "environmental_field_reports": ["document_date"],
}

format_standard = r"^\d{4}-\d{2}-\d{2}"
for nom, colonnes in colonnes_dates.items():
    for c in colonnes:
        autres = tables[nom][c].dropna()
        autres = autres[~autres.str.match(format_standard)]
        if len(autres):
            print(f"{nom} / {c} : {len(autres)} dates -> {autres.unique()[:6].tolist()}")

bat_surveys / survey_end : 10 dates -> ['12/01/2022 00:00:00', '12/01/2020 00:00:00', '2020/07/12 00:00:00', '2021/01/12 00:00:00', '2021/08/12 00:00:00', '12/07/2021 00:00:00']
monitoring_device_operations / operation_end : 20 dates -> ['31-May-2025 23:59:59', '31-Oct-2021 23:59:59', '30-Sep-2024 23:59:59', '2022/11/30 23:59:59', '30-Apr-2024 23:59:59', '2019/04/30 23:59:59']


Seules deux colonnes posent problème, `survey_end` et `operation_end`, avec trois autres formats : `12/01/2022`, `2021/01/12` et `12-Apr-2019`.

Le format `12/01/2022` est ambigu : est-ce le 12 janvier ou le 1er décembre ? Pour trancher, on s'appuie sur la date de début du même relevé, qui, elle, est toujours bien écrite. Dans ce fichier, un relevé commence le 5 du mois et se termine le 12 du même mois. Si `12/01/2022` correspond à un relevé commencé en janvier, alors le premier nombre est le jour.

In [11]:
chauves = tables["bat_surveys"]
ambigues = chauves[chauves["survey_end"].str.match(r"^\d{2}/\d{2}/\d{4}")]
ambigues[["survey_start", "survey_end"]]

,survey_start,survey_end
114,2022-01-05T18:00:00,12/01/2022 00:00:00
246,2020-01-05T18:00:00,12/01/2020 00:00:00
1200,2021-07-05T18:00:00,12/07/2021 00:00:00
3041,2025-06-05T18:00:00,12/06/2025 00:00:00


À chaque fois, le mois de début correspond au deuxième nombre : `12/01/2022` est donc bien le 12 janvier 2022, au format jour/mois/année. Pour les appareils, les fins de période sont des derniers jours du mois (« 30/04/2019 »), ce qui confirme le même ordre, puisqu'il n'existe pas de 30e mois.

On convertit donc chaque format avec la bonne règle.

In [12]:
def en_date(serie):
    texte = serie.astype("string").str.strip()
    resultat = pd.Series(pd.NaT, index=serie.index, dtype="datetime64[ns]")
    for forme in ["%Y-%m-%dT%H:%M:%S", "%Y-%m-%d", "%d/%m/%Y %H:%M:%S",
                  "%Y/%m/%d %H:%M:%S", "%d-%b-%Y %H:%M:%S"]:
        manque = resultat.isna()
        resultat[manque] = pd.to_datetime(texte[manque], format=forme, errors="coerce")
    return resultat

for nom, colonnes in colonnes_dates.items():
    for c in colonnes:
        avant = tables[nom][c].notna().sum()
        tables[nom][c] = en_date(tables[nom][c])
        perdues = avant - tables[nom][c].notna().sum()
        if perdues:
            print("dates non lues :", nom, c, perdues)
print("Conversion terminée")

Conversion terminée


Toutes les dates sont lues. On vérifie que les relevés de chauves-souris durent bien une semaine, et que les périodes des appareils couvrent un mois.

In [13]:
duree = (chauves["survey_end"] - chauves["survey_start"]).dt.days
print("Durée des relevés de chauves-souris (jours) :", duree.value_counts().to_dict())

appareils = tables["monitoring_device_operations"]
duree = (appareils["operation_end"] - appareils["operation_start"]).dt.days
print("Durée des périodes des appareils (jours) :", duree.value_counts().sort_index().to_dict())

Durée des relevés de chauves-souris (jours) : {6: 4992}
Durée des périodes des appareils (jours) : {27: 320, 28: 128, 29: 1664, 30: 2880}


Les durées sont cohérentes : 6 jours pour les relevés (du 5 au 12 du mois) et de 27 à 30 jours pour les appareils, selon la longueur du mois. Si on s'était trompé d'ordre jour/mois, on aurait vu apparaître des durées négatives ou de plusieurs mois.

## 5. Les catégories écrites de plusieurs façons

Une même catégorie peut être écrite avec des majuscules différentes ou des espaces en trop. Pour l'ordinateur, « Fixed_Acoustic_Station » et « fixed_acoustic_station » sont alors deux catégories différentes. On affiche les colonnes concernées.

In [14]:
for nom, df in tables.items():
    for c in df.select_dtypes(include=["object", "string"]).columns:
        v = df[c].dropna()
        propre = v.str.strip().str.lower().str.replace(r"\s+", " ", regex=True)
        if v.nunique() > propre.nunique():
            print(f"{nom} / {c} : {v.value_counts().to_dict()}\n")

monitoring_sites / site_category : {'habitat monitoring': 40, 'orchard monitoring': 15, 'cave monitoring': 8, 'ORCHARD MONITORING': 1}



bat_surveys / survey_method : {'acoustic_and_visual': 1742, 'acoustic_transect': 1616, 'fixed_acoustic_station': 1614, 'FIXED_ACOUSTIC_STATION': 9, 'ACOUSTIC_AND_VISUAL': 4, 'ACOUSTIC_TRANSECT': 4, 'Fixed_Acoustic_Station': 2, 'Acoustic_Transect': 1}

environmental_monthly / land_use_category : {'mixed_forest': 1481, 'coastal_habitat': 780, 'durian_orchard': 777, 'karst_habitat': 697, 'research_site': 467, 'mixed_agriculture': 388, 'protected_forest': 234, 'degraded_land': 153, 'Karst_Habitat': 4, 'DEGRADED_LAND': 2, 'DURIAN_ORCHARD': 2, 'Degraded_Land': 1, 'KARST_HABITAT': 1, 'Research_Site': 1, 'Durian_Orchard': 1, 'MIXED_AGRICULTURE': 1, 'Mixed_Agriculture': 1, 'Mixed_Forest': 1}

monitoring_device_operations / maintenance_type : {'routine inspection': 4783, 'corrective maintenance': 137, 'sensor replacement': 52, 'ROUTINE INSPECTION': 12, 'Routine Inspection': 8}

monitoring_device_operations / record_status : {'closed': 4972, 'CLOSED': 13, 'Closed': 7}

orchard_surveys / agricultu

Six colonnes sont concernées. Ce sont toutes des catégories en minuscules à l'origine (méthode de relevé, type de terrain, type de maintenance…), avec quelques variantes en majuscules ou avec des espaces en trop. On les remet en minuscules et on retire les espaces superflus.

On ne touche pas aux codes comme `site_code`, `device_reference` ou `fault_code` : ils sont écrits en majuscules partout et servent à relier les fichiers entre eux.

In [15]:
categories = {
    "monitoring_sites": ["site_category"],
    "bat_surveys": ["survey_method"],
    "environmental_monthly": ["land_use_category"],
    "monitoring_device_operations": ["maintenance_type", "record_status"],
    "orchard_surveys": ["agricultural_disturbance_note"],
}
for nom, colonnes in categories.items():
    for c in colonnes:
        tables[nom][c] = tables[nom][c].str.strip().str.lower().str.replace(r"\s+", " ", regex=True)
        print(f"{nom} / {c} : {tables[nom][c].value_counts().to_dict()}")

monitoring_sites / site_category : {'habitat monitoring': 40, 'orchard monitoring': 16, 'cave monitoring': 8}
bat_surveys / survey_method : {'acoustic_and_visual': 1746, 'fixed_acoustic_station': 1625, 'acoustic_transect': 1621}
environmental_monthly / land_use_category : {'mixed_forest': 1482, 'durian_orchard': 780, 'coastal_habitat': 780, 'karst_habitat': 702, 'research_site': 468, 'mixed_agriculture': 390, 'protected_forest': 234, 'degraded_land': 156}
monitoring_device_operations / maintenance_type : {'routine inspection': 4803, 'corrective maintenance': 137, 'sensor replacement': 52}
monitoring_device_operations / record_status : {'closed': 4992}
orchard_surveys / agricultural_disturbance_note : {'storm-related flower loss': 18, 'elevated pesticide application': 5}


## 6. Les valeurs manquantes

On compte les valeurs manquantes de chaque colonne, puis on décide quoi en faire.

In [16]:
manquantes = []
for nom, df in tables.items():
    for c, n in df.isna().sum().items():
        if n:
            manquantes.append({"fichier": nom, "colonne": c, "manquantes": n,
                               "part (%)": round(100 * n / len(df), 1)})
pd.DataFrame(manquantes)

,fichier,colonne,manquantes,part (%)
0,monitoring_sites,elevation_m,1,1.6
1,monitoring_sites,programme_end_date,64,100.0
2,bat_surveys,observer_team,20,0.4
3,environmental_monthly,vegetation_cover_pct,10,0.2
4,environmental_monthly,pesticide_index,15,0.3
5,environmental_monthly,weather_exception_code,4788,95.9
6,monitoring_device_operations,fault_code,4803,96.2
7,monitoring_device_operations,maintenance_provider,30,0.6
8,orchard_surveys,agricultural_disturbance_note,1225,98.2
9,environmental_field_reports,author_name,1,3.3


Il y a trois cas différents.

Une colonne entièrement vide : `programme_end_date`. Tous les sites sont encore actifs (`site_status` vaut « active » partout), donc aucun programme n'est terminé. Cette colonne n'apporte rien : on la supprime.

Des colonnes où une case vide veut dire « rien à signaler » : `weather_exception_code` (pas d'événement météo particulier), `fault_code` (pas de panne) et `agricultural_disturbance_note` (pas de perturbation notée). Ici, l'absence est une information. On garde les cases vides telles quelles.

Des oublis isolés : l'équipe d'observation, le prestataire de maintenance, la végétation, les pesticides, l'altitude d'un site ou l'auteur d'un rapport. Ils ne touchent que quelques lignes : au plus 30 sur près de 5 000 pour les gros fichiers, un site sur 64 pour l'altitude et un rapport sur 30 pour l'auteur. On ne supprime pas ces lignes, parce qu'on perdrait aussi le reste du relevé, notamment le nombre de chauves-souris. On ne les remplit pas non plus maintenant : si une analyse a besoin d'une valeur, elle choisira elle-même comment la compléter.

In [17]:
tables["monitoring_sites"] = tables["monitoring_sites"].drop(columns="programme_end_date")

## 7. Les liens entre les fichiers

Tous les fichiers renvoient à un site par la colonne `site_code`. On vérifie que chaque code utilisé existe bien dans la liste des sites, sinon les relevés concernés ne pourraient être rattachés à rien.

In [18]:
codes_connus = set(tables["monitoring_sites"]["site_code"])
for nom, df in tables.items():
    inconnus = set(df["site_code"]) - codes_connus
    print(f"{nom:30s} {df['site_code'].nunique():3d} sites utilisés, inconnus : {len(inconnus)}")

# Les rapports de terrain renvoient aussi à un relevé et à un appareil précis
rapports = tables["environmental_field_reports"]
print("Relevés cités introuvables :",
      (~rapports["related_survey_id"].isin(chauves["survey_id"])).sum())
print("Appareils cités introuvables :",
      (~rapports["related_device_reference"].isin(appareils["device_reference"])).sum())

monitoring_sites                64 sites utilisés, inconnus : 0
bat_surveys                     64 sites utilisés, inconnus : 0
environmental_monthly           64 sites utilisés, inconnus : 0
monitoring_device_operations    64 sites utilisés, inconnus : 0
orchard_surveys                 16 sites utilisés, inconnus : 0
environmental_field_reports     16 sites utilisés, inconnus : 0
Relevés cités introuvables : 0
Appareils cités introuvables : 0


Tous les liens fonctionnent : chaque site, chaque relevé et chaque appareil cités existent.

On remarque au passage que les rapports de terrain ne portent que sur 16 sites, et que leurs codes commencent tous par `KNT`, c'est-à-dire des sites de la région de Kintara. On le note pour l'analyse.

Le texte des rapports n'a pas été modifié : on garde ses majuscules et ses retours à la ligne, qui servent à repérer les parties des rapports (« INCIDENT », « EVIDENCE »…) pour l'analyse des textes.

## 8. Enregistrement des fichiers propres

On enregistre chaque table sous son nom d'origine dans `data_clean/ecology/`. Les notebooks d'analyse liront ces fichiers.

In [19]:
SORTIE.mkdir(parents=True, exist_ok=True)
for nom, df in tables.items():
    df.to_csv(SORTIE / f"{nom}.csv", index=False)
    print(f"{nom:30s} {len(df):5d} lignes enregistrées")

monitoring_sites                  64 lignes enregistrées


bat_surveys                     4992 lignes enregistrées


environmental_monthly           4992 lignes enregistrées
monitoring_device_operations    4992 lignes enregistrées
orchard_surveys                 1248 lignes enregistrées
environmental_field_reports       30 lignes enregistrées


## Bilan du nettoyage

| Problème | Où | Ce qu'on a fait |
|---|---|---|
| Lignes copiées deux fois | 4 fichiers, 2 lignes chacun | Supprimées |
| Virgule au lieu du point, unité collée au nombre | Comptages, pluie, surface, récolte… | Converties en nombres, aucune valeur perdue |
| Taux de nouaison écrits en fraction | 8 lignes de `orchard_surveys` | Multipliés par 100 |
| Dates en plusieurs formats | Fins de relevé et fins de période des appareils | Converties ; l'ordre jour/mois a été vérifié avec les dates de début |
| Catégories écrites de plusieurs façons | 6 colonnes | Mises en minuscules, espaces retirés |
| Colonne entièrement vide | `programme_end_date` | Supprimée |
| Cases vides qui veulent dire « rien à signaler » | Codes météo, pannes, perturbations | Gardées telles quelles |
| Petits oublis isolés | Quelques lignes par fichier | Gardés vides, sans supprimer les lignes |

Les données sont maintenant prêtes pour l'analyse.